<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/First_Model_Code_with_four_heads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ECGFOUNDER — MODEL SETUP
# CELL 1: DOWNLOAD PRETRAINED 12-LEAD MODEL
# ============================================================

import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "ECGFounder"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "12_lead_ECGFounder.pth"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Official Hugging Face download URL
MODEL_URL = (
    "https://huggingface.co/PKUDigitalHealth/ECGFounder/"
    "resolve/main/12_lead_ECGFounder.pth"
)

# ------------------------------------------------------------
# DOWNLOAD
# ------------------------------------------------------------

# if os.path.exists(MODEL_PATH):

#     print("ECGFounder model already exists.")
#     print("Path:", MODEL_PATH)

# else:

#     print("Downloading ECGFounder...")
#     print("Size: approximately 370 MB")
#     print()

#     response = requests.get(
#         MODEL_URL,
#         stream=True
#     )

#     response.raise_for_status()

#     total_size = int(
#         response.headers.get("content-length", 0)
#     )

#     with open(MODEL_PATH, "wb") as f:

#         with tqdm(
#             total=total_size,
#             unit="B",
#             unit_scale=True,
#             desc="ECGFounder"
#         ) as pbar:

#             for chunk in response.iter_content(
#                 chunk_size=1024 * 1024
#             ):

#                 if chunk:

#                     f.write(chunk)
#                     pbar.update(len(chunk))

#     print("\nDownload complete.")

# # ------------------------------------------------------------
# # VERIFY FILE
# # ------------------------------------------------------------

# file_size_mb = (
#     os.path.getsize(MODEL_PATH)
#     / (1024 ** 2)
# )

# print("\n" + "=" * 60)
# print("ECGFOUNDER DOWNLOAD CHECK")
# print("=" * 60)

# print("File:")
# print(MODEL_PATH)

# print(f"\nSize: {file_size_mb:.2f} MB")

# print(
#     "Exists:",
#     os.path.exists(MODEL_PATH)
# )

# print("=" * 60)

Mounted at /content/drive


In [ ]:
# ============================================================
# ECGFOUNDER — CELL 2
# INSPECT PRETRAINED CHECKPOINT
# ============================================================

import torch
import os

MODEL_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "ECGFounder/12_lead_ECGFounder.pth"
)

print("=" * 70)
print("LOADING ECGFOUNDER CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:")
print(type(checkpoint))

# ------------------------------------------------------------
# INSPECT TOP-LEVEL CONTENT
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")

    for key in checkpoint.keys():
        print(" -", key)

else:

    print("\nCheckpoint is not a dictionary.")

# ------------------------------------------------------------
# BASIC INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT INSPECTION COMPLETE")
print("=" * 70)

LOADING ECGFOUNDER CHECKPOINT

Checkpoint type:
<class 'dict'>

Checkpoint keys:
 - epoch
 - step
 - state_dict
 - optimizer
 - scheduler
 - val_auroc

CHECKPOINT INSPECTION COMPLETE


In [ ]:
# ============================================================
# ECGFOUNDER — CELL 3
# INSPECT MODEL WEIGHTS
# ============================================================

state_dict = checkpoint["state_dict"]

print("=" * 70)
print("ECGFOUNDER STATE DICTIONARY")
print("=" * 70)

print("\nNumber of parameter tensors:", len(state_dict))

print("\nFirst 30 parameter names:\n")

for i, (name, tensor) in enumerate(state_dict.items()):

    print(
        f"{i+1:2d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )

    if i == 29:
        break



# ============================================================
# ECGFOUNDER — CELL 4
# INSPECT FINAL MODEL LAYERS
# ============================================================

print("=" * 70)
print("LAST 40 ECGFOUNDER PARAMETERS")
print("=" * 70)

items = list(state_dict.items())

for i, (name, tensor) in enumerate(items[-40:], start=len(items) - 39):

    print(
        f"{i:3d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )


ECGFOUNDER STATE DICTIONARY

Number of parameter tensors: 509

First 30 parameter names:

 1. first_conv.conv.weight                                                 shape=(64, 12, 16)
 2. first_conv.conv.bias                                                   shape=(64,)
 3. first_bn.weight                                                        shape=(64,)
 4. first_bn.bias                                                          shape=(64,)
 5. first_bn.running_mean                                                  shape=(64,)
 6. first_bn.running_var                                                   shape=(64,)
 7. first_bn.num_batches_tracked                                           shape=()
 8. stage_list.0.block_list.0.bn1.weight                                   shape=(64,)
 9. stage_list.0.block_list.0.bn1.bias                                     shape=(64,)
10. stage_list.0.block_list.0.bn1.running_mean                             shape=(64,)
11. stage_list.0.block_list.0.bn1.ru

In [ ]:
import torch
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "/content/drive/MyDrive/ECG_Project/ECGFounder/12_lead_ECGFounder.pth"

import numpy as np
from collections import Counter
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __getitem__(self, index):
        return (torch.tensor(self.data[index], dtype=torch.float), torch.tensor(self.label[index], dtype=torch.long))

    def __len__(self):
        return len(self.data)

class MyConv1dPadSame(nn.Module):
    """
    extend nn.Conv1d to support SAME padding

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super(MyConv1dPadSame, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.conv = torch.nn.Conv1d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

    def forward(self, x):

        net = x

        # compute pad shape
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        # print(net.shape)
        net = self.conv(net)

        return net

class MyMaxPool1dPadSame(nn.Module):
    """
    extend nn.MaxPool1d to support SAME padding

    params:
        kernel_size: kernel size
        stride: the stride of the window. Default value is kernel_size

    input: (n_sample, n_channel, n_length)
    """
    def __init__(self, kernel_size):
        super(MyMaxPool1dPadSame, self).__init__()
        self.kernel_size = kernel_size
        self.max_pool = torch.nn.MaxPool1d(kernel_size=self.kernel_size)

    def forward(self, x):

        net = x

        # compute pad shape
        p = max(0, self.kernel_size - 1)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)

        net = self.max_pool(net)

        return net

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class BasicBlock(nn.Module):
    """
    Basic Block:
        conv1 -> convk -> conv1

    params:
        in_channels: number of input channels
        out_channels: number of output channels
        ratio: ratio of channels to out_channels
        kernel_size: kernel window length
        stride: kernel step size
        groups: number of groups in convk
        downsample: whether downsample length
        use_bn: whether use batch_norm
        use_do: whether use dropout

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, downsample, is_first_block=False, use_bn=True, use_do=True):
        super(BasicBlock, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.downsample = downsample
        self.stride = stride if self.downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.middle_channels = int(self.out_channels * self.ratio)

        # the first conv, conv1
        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(
            in_channels=self.in_channels,
            out_channels=self.middle_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # the second conv, convk
        self.bn2 = nn.BatchNorm1d(self.middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.middle_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

        # the third conv, conv1
        self.bn3 = nn.BatchNorm1d(self.middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.out_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # Squeeze-and-Excitation
        r = 2
        self.se_fc1 = nn.Linear(self.out_channels, self.out_channels//r)
        self.se_fc2 = nn.Linear(self.out_channels//r, self.out_channels)
        self.se_activation = Swish()

        if self.downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):

        identity = x

        out = x
        # the first conv, conv1
        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)
        out = self.conv1(out)

        # the second conv, convk
        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)
        out = self.conv2(out)

        # the third conv, conv1
        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)
        out = self.conv3(out) # (n_sample, n_channel, n_length)

        # Squeeze-and-Excitation
        se = out.mean(-1) # (n_sample, n_channel)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se) # (n_sample, n_channel)
        out = torch.einsum('abc,ab->abc', out, se)

        # if downsample, also downsample identity
        if self.downsample:
            identity = self.max_pool(identity)

        # if expand channel, also pad zeros to identity
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1,-2)
            ch1 = (self.out_channels-self.in_channels)//2
            ch2 = self.out_channels-self.in_channels-ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1,-2)

        # shortcut
        out += identity

        return out

class BasicStage(nn.Module):
    """
    Basic Stage:
        block_1 -> block_2 -> ... -> block_M
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, i_stage, m_blocks, use_bn=True, use_do=True, verbose=False):
        super(BasicStage, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.i_stage = i_stage
        self.m_blocks = m_blocks
        self.use_bn = use_bn
        self.use_do = use_do
        self.verbose = verbose

        self.block_list = nn.ModuleList()
        for i_block in range(self.m_blocks):

            # first block
            if self.i_stage == 0 and i_block == 0:
                self.is_first_block = True
            else:
                self.is_first_block = False
            # downsample, stride, input
            if i_block == 0:
                self.downsample = True
                self.stride = stride
                self.tmp_in_channels = self.in_channels
            else:
                self.downsample = False
                self.stride = 1
                self.tmp_in_channels = self.out_channels

            # build block
            tmp_block = BasicBlock(
                in_channels=self.tmp_in_channels,
                out_channels=self.out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=self.groups,
                downsample=self.downsample,
                is_first_block=self.is_first_block,
                use_bn=self.use_bn,
                use_do=self.use_do)
            self.block_list.append(tmp_block)

    def forward(self, x):

        out = x

        for i_block in range(self.m_blocks):
            net = self.block_list[i_block]
            out = net(out)
            if self.verbose:
                print('stage: {}, block: {}, in_channels: {}, out_channels: {}, outshape: {}'.format(self.i_stage, i_block, net.in_channels, net.out_channels, list(out.shape)))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv1.in_channels, net.conv1.out_channels, net.conv1.kernel_size, net.conv1.stride, net.conv1.groups))
                print('stage: {}, block: {}, convk: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv2.in_channels, net.conv2.out_channels, net.conv2.kernel_size, net.conv2.stride, net.conv2.groups))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv3.in_channels, net.conv3.out_channels, net.conv3.kernel_size, net.conv3.stride, net.conv3.groups))

        return out

class Net1D(nn.Module):
    """

    Input:
        X: (n_samples, n_channel, n_length)
        Y: (n_samples)

    Output:
        out: (n_samples)

    params:
        in_channels
        base_filters
        filter_list: list, filters for each stage
        m_blocks_list: list, number of blocks of each stage
        kernel_size
        stride
        groups_width
        n_stages
        n_classes
        use_bn
        use_do

    """

    def __init__(self, in_channels, base_filters, ratio, filter_list, m_blocks_list, kernel_size, stride, groups_width, n_classes, use_bn=True, use_do=True, return_features=False, verbose=False):
        super(Net1D, self).__init__()

        self.in_channels = in_channels
        self.base_filters = base_filters
        self.ratio = ratio
        self.filter_list = filter_list
        self.m_blocks_list = m_blocks_list
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups_width = groups_width
        self.n_stages = len(filter_list)
        self.n_classes = n_classes
        self.use_bn = use_bn
        self.use_do = use_do
        self.return_features = return_features
        self.verbose = verbose

        # first conv
        self.first_conv = MyConv1dPadSame(
            in_channels=in_channels,
            out_channels=self.base_filters,
            kernel_size=self.kernel_size,
            stride=2)
        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        # stages
        self.stage_list = nn.ModuleList()
        in_channels = self.base_filters
        for i_stage in range(self.n_stages):

            out_channels = self.filter_list[i_stage]
            m_blocks = self.m_blocks_list[i_stage]
            tmp_stage = BasicStage(
                in_channels=in_channels,
                out_channels=out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=out_channels//self.groups_width,
                i_stage=i_stage,
                m_blocks=m_blocks,
                use_bn=self.use_bn,
                use_do=self.use_do,
                verbose=self.verbose)
            self.stage_list.append(tmp_stage)
            in_channels = out_channels

        # final prediction
        self.dense = nn.Linear(in_channels, n_classes)

    def forward(self, x):

        out = x

        # first conv
        out = self.first_conv(out)
        if self.use_bn:
            out = self.first_bn(out)
        out = self.first_activation(out)

        # stages
        for i_stage in range(self.n_stages):
            net = self.stage_list[i_stage]
            out = net(out)

        # final prediction
        deep_features = out.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features
        else:
            return out

if __name__ == "__main__":
  model = Net1D(
      in_channels=12,
      base_filters=64, #32 64
      ratio=1,
      filter_list=[64,160,160,400,400,1024,1024],    #[16,32,32,80,80,256,256] [32,64,64,160,160,512,512] [64,160,160,400,400,1024,1024]
      m_blocks_list=[2,2,2,3,3,4,4],   #[2,2,2,2,2,2,2] [2,2,2,3,3,4,4]
      kernel_size=16,
      stride=2,
      groups_width=16,
      verbose=True,
      use_bn=False,
      use_do=False,
      n_classes=150)
  print(model)



Net1D(
  (first_conv): MyConv1dPadSame(
    (conv): Conv1d(12, 64, kernel_size=(16,), stride=(2,))
  )
  (first_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_activation): Swish()
  (stage_list): ModuleList(
    (0): BasicStage(
      (block_list): ModuleList(
        (0): BasicBlock(
          (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation1): Swish()
          (do1): Dropout(p=0.5, inplace=False)
          (conv1): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
          )
          (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation2): Swish()
          (do2): Dropout(p=0.5, inplace=False)
          (conv2): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(16,), stride=(2,), groups=4)
          )
          (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [ ]:
# Build exact ECGFounder architecture
model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=150,
    use_bn=False,
    use_do=False,
    return_features=True,
    verbose=False
)

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["state_dict"])

model = model.to(DEVICE)

print("✓ ECGFounder loaded successfully")
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

✓ ECGFounder loaded successfully
Device: cuda
Parameters: 30819014


In [ ]:
# ============================================================
# CELL — LABEL MAPPING / TARGET DEFINITION
# ============================================================

import pandas as pd

CSV_PATH = "/content/drive/MyDrive/ECG_Project/results/dataset_split/train_split.csv"

df = pd.read_csv(CSV_PATH)

# ------------------------------------------------------------
# 1. LABELS PRESENT IN DATASET
# ------------------------------------------------------------

LABELS = [
    "AMI", "OMI", "CTO", "PCI",
    "NSTEMI", "STEMI", "UA",
    "LM", "PLAD", "MLAD", "DLAD", "DB",
    "PLCX", "MLCX", "DLCX", "OM",
    "PRCA", "MRCA", "DRCA",
    "VF_VT", "Paced", "Prior_PCI"
]

print("Total labels found:", len(LABELS))

missing = [x for x in LABELS if x not in df.columns]

if missing:
    print("❌ Missing labels:", missing)
else:
    print("✓ All 22 labels found")


# ============================================================
# 2. OUR FOUR HEADS
# ============================================================

ACS_LABELS = [
    "STEMI",
    "NSTEMI",
    "UA"
]

SEVERITY_LABELS = [
    "OMI",
    "CTO"
]

CORONARY_LABELS = [
    "LM",
    "PLAD",
    "MLAD",
    "DLAD",
    "DB",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

CONTEXT_LABELS = [
    "Paced",
    "Prior_PCI",
    "VF_VT"
]


# ============================================================
# 3. VERIFY LABEL COUNTS
# ============================================================

print("\n================ LABEL GROUPS ================\n")

print("ACS HEAD:")
print(ACS_LABELS)

print("\nSEVERITY HEAD:")
print(SEVERITY_LABELS)

print("\nCORONARY HEAD:")
print(CORONARY_LABELS)

print("\nCONTEXT HEAD:")
print(CONTEXT_LABELS)

print("\nTotal assigned:",
      len(ACS_LABELS)
      + len(SEVERITY_LABELS)
      + len(CORONARY_LABELS)
      + len(CONTEXT_LABELS))


# ============================================================
# 4. CREATE TARGET TABLE
# ============================================================

targets = df[
    ACS_LABELS +
    SEVERITY_LABELS +
    CORONARY_LABELS +
    CONTEXT_LABELS
].copy()

print("\nTarget matrix shape:", targets.shape)

print("\n✓ LABEL MAPPING COMPLETE")

Total labels found: 22
✓ All 22 labels found

================ LABEL GROUPS ================

ACS HEAD:
['STEMI', 'NSTEMI', 'UA']

SEVERITY HEAD:
['OMI', 'CTO']

CORONARY HEAD:
['LM', 'PLAD', 'MLAD', 'DLAD', 'DB', 'PLCX', 'MLCX', 'DLCX', 'OM', 'PRCA', 'MRCA', 'DRCA']

CONTEXT HEAD:
['Paced', 'Prior_PCI', 'VF_VT']

Total assigned: 20

Target matrix shape: (15275, 20)

✓ LABEL MAPPING COMPLETE


In [ ]:
# ============================================================
# MULTI-HEAD ACS MODEL
# ============================================================

class MultiHeadECG(nn.Module):

    def __init__(self, feature_dim=1024):
        super().__init__()

        # ACS Head
        # 0 = Non-ACS
        # 1 = STEMI
        # 2 = NSTEMI
        # 3 = UA
        self.acs_head = nn.Linear(feature_dim, 4)

        # Severity Head
        # 0 = None
        # 1 = OMI
        # 2 = CTO
        self.severity_head = nn.Linear(feature_dim, 3)

        # Coronary Head
        # 10 vessels × 3 states
        self.coronary_head = nn.Linear(feature_dim, 10 * 3)

        # Context Head
        # Paced, Prior_PCI, VF_VT
        self.context_head = nn.Linear(feature_dim, 3)

    def forward(self, features):

        acs = self.acs_head(features)
        severity = self.severity_head(features)
        coronary = self.coronary_head(features)
        context = self.context_head(features)

        # Reshape coronary output:
        # (batch, 30) → (batch, 10 vessels, 3 states)
        coronary = coronary.view(
            features.size(0), 10, 3
        )

        return {
            "acs": acs,
            "severity": severity,
            "coronary": coronary,
            "context": context
        }


# Create heads
heads = MultiHeadECG(feature_dim=1024)

print(heads)

MultiHeadECG(
  (acs_head): Linear(in_features=1024, out_features=4, bias=True)
  (severity_head): Linear(in_features=1024, out_features=3, bias=True)
  (coronary_head): Linear(in_features=1024, out_features=30, bias=True)
  (context_head): Linear(in_features=1024, out_features=3, bias=True)
)


In [ ]:
class ECGFounderMultiHead(nn.Module):

    def __init__(self, backbone, feature_dim=1024):
        super().__init__()

        self.backbone = backbone

        # We only need ECGFounder's 1024-D features
        self.backbone.return_features = True

        # -------------------------
        # ACS HEAD
        # 0 = Non-ACS
        # 1 = STEMI
        # 2 = NSTEMI
        # 3 = UA
        # -------------------------
        self.acs_head = nn.Linear(feature_dim, 4)

        # -------------------------
        # SEVERITY HEAD
        # 0 = None
        # 1 = OMI
        # 2 = CTO
        # -------------------------
        self.severity_head = nn.Linear(feature_dim, 3)

        # -------------------------
        # CORONARY HEAD
        # 10 vessels × 3 states
        # -------------------------
        self.coronary_head = nn.Linear(feature_dim, 30)

        # -------------------------
        # CONTEXT HEAD
        # Paced, Prior_PCI, VF_VT
        # -------------------------
        self.context_head = nn.Linear(feature_dim, 3)

    def forward(self, x):

        # ECGFounder
        _, features = self.backbone(x)

        # Four parallel heads
        acs = self.acs_head(features)
        severity = self.severity_head(features)
        coronary = self.coronary_head(features)
        context = self.context_head(features)

        # 30 → 10 vessels × 3 states
        coronary = coronary.view(
            x.size(0), 10, 3
        )

        return acs, severity, coronary, context


# ============================================================
# CREATE COMPLETE MODEL
# ============================================================

multi_model = ECGFounderMultiHead(model)

multi_model = multi_model.to(DEVICE)
multi_model.eval()


# ============================================================
# TEST WITH DUMMY ECG
# ============================================================

x = torch.randn(2, 12, 5000).to(DEVICE)

with torch.no_grad():

    acs, severity, coronary, context = multi_model(x)


print("============================================================")
print("MULTI-HEAD OUTPUT VERIFICATION")
print("============================================================")

print("Input       :", x.shape)
print("ACS         :", acs.shape)
print("Severity    :", severity.shape)
print("Coronary    :", coronary.shape)
print("Context     :", context.shape)

print("============================================================")
print("✓ MODEL CONNECTION VERIFIED")
print("============================================================")

MULTI-HEAD OUTPUT VERIFICATION
Input       : torch.Size([2, 12, 5000])
ACS         : torch.Size([2, 4])
Severity    : torch.Size([2, 3])
Coronary    : torch.Size([2, 10, 3])
Context     : torch.Size([2, 3])
✓ MODEL CONNECTION VERIFIED


In [ ]:
# ============================================================
# FREEZE ECGFOUNDER — TRAIN ONLY THE 4 HEADS
# ============================================================

# Freeze ECGFounder
for param in multi_model.backbone.parameters():
    param.requires_grad = False

# Keep the four heads trainable
for param in multi_model.acs_head.parameters():
    param.requires_grad = True

for param in multi_model.severity_head.parameters():
    param.requires_grad = True

for param in multi_model.coronary_head.parameters():
    param.requires_grad = True

for param in multi_model.context_head.parameters():
    param.requires_grad = True


# Check trainable parameters
trainable = sum(
    p.numel()
    for p in multi_model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in multi_model.parameters()
)

print("============================================================")
print("PARAMETER FREEZING")
print("============================================================")
print(f"Total parameters     : {total:,}")
print(f"Trainable parameters : {trainable:,}")
print(f"Frozen parameters    : {total - trainable:,}")
print("============================================================")

PARAMETER FREEZING
Total parameters     : 30,860,014
Trainable parameters : 41,000
Frozen parameters    : 30,819,014


In [ ]:
# ============================================================
# REMOVE THE 2 KNOWN CORRUPTED ECGs FROM TRAIN LABEL CSV
# ============================================================

import pandas as pd
import os

BASE = "/content/drive/MyDrive/ECG_Project"

TRAIN_CSV = os.path.join(
    BASE,
    "results/dataset_split/train_split.csv"
)

# Known corrupted ECG records
CORRUPTED_RECORDS = [
    "14262.dat",
    "03228.dat"
]

df = pd.read_csv(TRAIN_CSV)

print("Original CSV rows:", len(df))

# Remove corrupted records using ecg_row_record
df_clean = df[
    ~df["ecg_row_record"].astype(str).isin(CORRUPTED_RECORDS)
].reset_index(drop=True)

print("Removed rows:", len(df) - len(df_clean))
print("Clean CSV rows:", len(df_clean))

print("\nRemoved records:")
print(
    df[
        df["ecg_row_record"].astype(str).isin(CORRUPTED_RECORDS)
    ][["Patient_id", "ecg_row_record"]]
)

# ============================================================
# VERIFY
# ============================================================

assert len(df_clean) == 15273

print("\n✓ TRAIN LABELS NOW MATCH H5:")
print("H5 signals : 15273")
print("CSV labels : 15273")

Original CSV rows: 15275
Removed rows: 2
Clean CSV rows: 15273

Removed records:
      Patient_id ecg_row_record
7163      P08823      14262.dat
12522     P15456      03228.dat

✓ TRAIN LABELS NOW MATCH H5:
H5 signals : 15273
CSV labels : 15273


In [ ]:
# ============================================================
# ACS19955 DATASET + DATALOADERS
# H5 WAVEFORMS + CLEAN CSV LABELS
#
# TARGET FORMAT:
# ACS:
#   0 = Non-ACS
#   1 = STEMI
#   2 = NSTEMI
#   3 = UA
#
# SEVERITY:
#   0 = None
#   1 = OMI
#   2 = CTO
#
# CORONARY:
#   10 vessels × 3 states
#
# CONTEXT:
#   Paced / Prior_PCI / VF_VT
# ============================================================

import os
import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. PATHS
# ============================================================

BASE = "/content/drive/MyDrive/ECG_Project"

TRAIN_H5 = os.path.join(
    BASE,
    "results/preprocessing/processed/train_fixed.h5"
)

VAL_H5 = os.path.join(
    BASE,
    "results/preprocessing/processed/validation.h5"
)

VAL_CSV = os.path.join(
    BASE,
    "results/dataset_split/validation_split.csv"
)


# ============================================================
# 2. CHECK REQUIRED VARIABLES
# ============================================================

print("=" * 60)
print("CHECKING DATASET SETUP")
print("=" * 60)

print("Train H5 exists :", os.path.exists(TRAIN_H5))
print("Val H5 exists   :", os.path.exists(VAL_H5))

if "df_clean" not in globals():
    raise NameError(
        "df_clean is not available.\n"
        "Run the cell that removed the 2 corrupted training "
        "records first."
    )

if "VAL_CSV" not in globals():
    raise NameError(
        "VAL_CSV is not defined.\n"
        "Set VAL_CSV to your validation labels CSV."
    )

print("Clean train CSV :", df_clean.shape)


# ============================================================
# 3. LABEL GROUPS
# ============================================================

# ACS is converted into ONE 4-class target
ACS_LABELS = [
    "STEMI",
    "NSTEMI",
    "UA"
]

# Severity is converted into ONE 3-class target
SEVERITY_LABELS = [
    "OMI",
    "CTO"
]

# LM and DB removed because of extremely low prevalence
CORONARY_LABELS = [
    "PLAD",
    "MLAD",
    "DLAD",
    "PLCX",
    "MLCX",
    "DLCX",
    "OM",
    "PRCA",
    "MRCA",
    "DRCA"
]

CONTEXT_LABELS = [
    "Paced",
    "Prior_PCI",
    "VF_VT"
]


# ============================================================
# 4. DATASET CLASS
# ============================================================

class ACS19955Dataset(Dataset):

    def __init__(self, h5_path, labels_df):

        self.h5_path = h5_path

        # Independent copy of labels
        self.labels_df = labels_df.reset_index(drop=True).copy()

        # ----------------------------------------------------
        # Open H5
        # ----------------------------------------------------

        self.h5 = h5py.File(h5_path, "r")

        self.signals = self.h5["signals"]
        self.record_ids = self.h5["record_ids"]

        print("\n" + "-" * 60)
        print("Loading:", os.path.basename(h5_path))
        print("-" * 60)

        print("H5 signals shape :", self.signals.shape)
        print("H5 records       :", len(self.record_ids))
        print("CSV rows         :", len(self.labels_df))

        # ----------------------------------------------------
        # Verify number of samples
        # ----------------------------------------------------

        if len(self.signals) != len(self.labels_df):

            raise ValueError(
                f"Number of H5 signals ({len(self.signals)}) "
                f"does not match CSV rows ({len(self.labels_df)})."
            )

        # ----------------------------------------------------
        # Verify required columns
        # ----------------------------------------------------

        required_columns = (
            ACS_LABELS
            + SEVERITY_LABELS
            + CORONARY_LABELS
            + CONTEXT_LABELS
        )

        missing_columns = [
            col
            for col in required_columns
            if col not in self.labels_df.columns
        ]

        if missing_columns:

            raise ValueError(
                "Missing label columns:\n"
                + str(missing_columns)
            )

        # ====================================================
        # ACS → 4 CLASS
        # ====================================================
        #
        # 0 = Non-ACS
        # 1 = STEMI
        # 2 = NSTEMI
        # 3 = UA
        #
        # ====================================================

        self.acs = np.zeros(
            len(self.labels_df),
            dtype=np.int64
        )

        stemi = self.labels_df["STEMI"].values.astype(int)
        nstemi = self.labels_df["NSTEMI"].values.astype(int)
        ua = self.labels_df["UA"].values.astype(int)

        self.acs[stemi == 1] = 1
        self.acs[nstemi == 1] = 2
        self.acs[ua == 1] = 3

        # Check for overlapping ACS labels
        acs_overlap = (
            stemi.astype(int)
            + nstemi.astype(int)
            + ua.astype(int)
        )

        if np.any(acs_overlap > 1):

            overlap_count = np.sum(acs_overlap > 1)

            raise ValueError(
                f"{overlap_count} samples have multiple "
                "ACS labels simultaneously."
            )

        # ====================================================
        # SEVERITY → 3 CLASS
        # ====================================================
        #
        # 0 = None
        # 1 = OMI
        # 2 = CTO
        #
        # ====================================================

        self.severity = np.zeros(
            len(self.labels_df),
            dtype=np.int64
        )

        omi = self.labels_df["OMI"].values.astype(int)
        cto = self.labels_df["CTO"].values.astype(int)

        self.severity[omi == 1] = 1
        self.severity[cto == 1] = 2

        # Check for overlapping severity labels
        severity_overlap = omi + cto

        if np.any(severity_overlap > 1):

            overlap_count = np.sum(
                severity_overlap > 1
            )

            raise ValueError(
                f"{overlap_count} samples have both "
                "OMI and CTO labels."
            )

        # ====================================================
        # CORONARY
        # ====================================================
        #
        # 10 vessels × 3 states
        #
        # 0 = none
        # 1 = acute
        # 2 = chronic
        #
        # ====================================================

        self.coronary = self.labels_df[
            CORONARY_LABELS
        ].values.astype(np.int64)

        # ====================================================
        # CONTEXT
        # ====================================================

        self.context = self.labels_df[
            CONTEXT_LABELS
        ].values.astype(np.float32)

        # ====================================================
        # FINAL VERIFICATION
        # ====================================================

        print("\nTARGET SHAPES")

        print(
            "ACS       :",
            self.acs.shape,
            "(4-class)"
        )

        print(
            "Severity  :",
            self.severity.shape,
            "(3-class)"
        )

        print(
            "Coronary  :",
            self.coronary.shape,
            "(10 vessels)"
        )

        print(
            "Context   :",
            self.context.shape,
            "(3 binary)"
        )

        print("✓ Dataset alignment verified")


    # ========================================================
    # GET ITEM
    # ========================================================

    def __getitem__(self, index):

        # ECG waveform
        signal = torch.tensor(
            self.signals[index],
            dtype=torch.float32
        )

        # ACS class
        acs = torch.tensor(
            self.acs[index],
            dtype=torch.long
        )

        # Severity class
        severity = torch.tensor(
            self.severity[index],
            dtype=torch.long
        )

        # Coronary:
        # 10 vessels × 3 states
        coronary = torch.tensor(
            self.coronary[index],
            dtype=torch.long
        )

        # Context:
        # Paced / Prior_PCI / VF_VT
        context = torch.tensor(
            self.context[index],
            dtype=torch.float32
        )

        return (
            signal,
            acs,
            severity,
            coronary,
            context
        )


    # ========================================================
    # LENGTH
    # ========================================================

    def __len__(self):

        return len(self.signals)


    # ========================================================
    # CLOSE H5
    # ========================================================

    def close(self):

        self.h5.close()


# ============================================================
# 5. LOAD VALIDATION CSV
# ============================================================

val_df = pd.read_csv(VAL_CSV)
print("\n" + "=" * 60)
print("VALIDATION CSV")
print("=" * 60)

print("Validation shape:", val_df.shape)


# ============================================================
# 6. CREATE TRAIN DATASET
# ============================================================

print("\n" + "=" * 60)
print("CREATING TRAIN DATASET")
print("=" * 60)

train_dataset = ACS19955Dataset(
    TRAIN_H5,
    df_clean
)


# ============================================================
# 7. CREATE VALIDATION DATASET
# ============================================================

print("\n" + "=" * 60)
print("CREATING VALIDATION DATASET")
print("=" * 60)

val_dataset = ACS19955Dataset(
    VAL_H5,
    val_df
)


# ============================================================
# 8. CREATE DATALOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ============================================================
# 9. DATALOADER VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("DATALOADER VERIFICATION")
print("=" * 60)

(
    ecg,
    acs,
    severity,
    coronary,
    context
) = next(iter(train_loader))

print("ECG       :", ecg.shape)
print("ACS       :", acs.shape)
print("Severity  :", severity.shape)
print("Coronary  :", coronary.shape)
print("Context   :", context.shape)


# ============================================================
# 10. VERIFY EXPECTED SHAPES
# ============================================================

assert ecg.shape[1:] == (12, 5000)

# ACS is now one class per ECG
assert acs.ndim == 1

# Severity is now one class per ECG
assert severity.ndim == 1

# 10 coronary vessels
assert coronary.shape[1:] == (10,)

# 3 binary context labels
assert context.shape[1:] == (3,)


# ============================================================
# 11. VERIFY CLASS RANGES
# ============================================================

assert torch.all(
    (acs >= 0) & (acs <= 3)
)

assert torch.all(
    (severity >= 0) & (severity <= 2)
)

assert torch.all(
    (coronary >= 0) & (coronary <= 2)
)


# ============================================================
# 12. SHOW TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("TRAIN TARGET DISTRIBUTION")
print("=" * 60)

print(
    "ACS classes:"
)

print(
    "  0 Non-ACS :",
    np.sum(train_dataset.acs == 0)
)

print(
    "  1 STEMI   :",
    np.sum(train_dataset.acs == 1)
)

print(
    "  2 NSTEMI  :",
    np.sum(train_dataset.acs == 2)
)

print(
    "  3 UA      :",
    np.sum(train_dataset.acs == 3)
)

print(
    "\nSeverity classes:"
)

print(
    "  0 None :",
    np.sum(train_dataset.severity == 0)
)

print(
    "  1 OMI  :",
    np.sum(train_dataset.severity == 1)
)

print(
    "  2 CTO  :",
    np.sum(train_dataset.severity == 2)
)


# ============================================================
# 13. FINAL STATUS
# ============================================================

print("\n" + "=" * 60)
print("✓ DATASET + DATALOADER READY")
print("=" * 60)

print(
    "Training samples   :",
    len(train_dataset)
)

print(
    "Validation samples :",
    len(val_dataset)
)

print(
    "Batch size         :",
    BATCH_SIZE
)

print(
    "ECG shape          :",
    "(12, 5000)"
)

print(
    "ACS outputs        :",
    4
)

print(
    "Severity outputs   :",
    3
)

print(
    "Coronary vessels   :",
    10
)

print(
    "Context outputs    :",
    3
)

print("=" * 60)

CHECKING DATASET SETUP
Train H5 exists : True
Val H5 exists   : True
Clean train CSV : (15273, 28)

VALIDATION CSV
Validation shape: (2685, 28)

CREATING TRAIN DATASET

------------------------------------------------------------
Loading: train_fixed.h5
------------------------------------------------------------
H5 signals shape : (15273, 12, 5000)
H5 records       : 15273
CSV rows         : 15273

TARGET SHAPES
ACS       : (15273,) (4-class)
Severity  : (15273,) (3-class)
Coronary  : (15273, 10) (10 vessels)
Context   : (15273, 3) (3 binary)
✓ Dataset alignment verified

CREATING VALIDATION DATASET

------------------------------------------------------------
Loading: validation.h5
------------------------------------------------------------
H5 signals shape : (2685, 12, 5000)
H5 records       : 2685
CSV rows         : 2685

TARGET SHAPES
ACS       : (2685,) (4-class)
Severity  : (2685,) (3-class)
Coronary  : (2685, 10) (10 vessels)
Context   : (2685, 3) (3 binary)
✓ Dataset alignmen

Stage 1 — Loss functions + optimizer(EXPERIMENTATION)

In [ ]:
# ============================================================
# STAGE 1 — CLASSIFIER HEAD TRAINING
# LOSS FUNCTIONS + OPTIMIZER
#
# ECGFounder backbone: FROZEN
# Only the 4 new classifier heads will train.
# ============================================================

import torch
import torch.nn as nn
import numpy as np

# ============================================================
# 0. DEVICE SETUP
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

multi_model = multi_model.to(device)

print("Device:", device)


# ============================================================
# 1. VERIFY BACKBONE IS FROZEN
# ============================================================

print("=" * 60)
print("CHECKING ECGFOUNDER FREEZE STATUS")
print("=" * 60)

backbone_trainable = sum(
    p.numel()
    for p in multi_model.backbone.parameters()
    if p.requires_grad
)

head_trainable = sum(
    p.numel()
    for name, p in multi_model.named_parameters()
    if p.requires_grad and "backbone" not in name
)

print("Backbone trainable parameters :", backbone_trainable)
print("Head trainable parameters     :", head_trainable)

assert backbone_trainable == 0, \
    "ECGFounder backbone is NOT fully frozen."

assert head_trainable > 0, \
    "Classifier heads are not trainable."

print("✓ ECGFounder backbone is frozen")
print("✓ Classifier heads are trainable")


# ============================================================
# 2. CREATE 4-CLASS ACS TARGET
# ============================================================
#
# Existing dataset stores:
#
# STEMI / NSTEMI / UA
#
# as three separate binary columns.
#
# Our ACS head has 4 classes:
#
# 0 = Non-ACS
# 1 = STEMI
# 2 = NSTEMI
# 3 = UA
#
# ============================================================

stemI   = train_dataset.acs[:, 0]
nstemI  = train_dataset.acs[:, 1]
ua      = train_dataset.acs[:, 2]

acs_target = np.zeros(
    len(train_dataset),
    dtype=np.int64
)

# STEMI
acs_target[stemI == 1] = 1

# NSTEMI
acs_target[nstemI == 1] = 2

# UA
acs_target[ua == 1] = 3


# ------------------------------------------------------------
# Verify no sample has multiple ACS labels
# ------------------------------------------------------------

acs_sum = (
    train_dataset.acs[:, 0]
    + train_dataset.acs[:, 1]
    + train_dataset.acs[:, 2]
)

multiple_acs = np.sum(acs_sum > 1)

print("\nACS TARGET VERIFICATION")
print("-----------------------")
print("Samples with multiple ACS labels :", multiple_acs)

assert multiple_acs == 0, \
    "Some samples contain multiple ACS labels. " \
    "The 4-class ACS target cannot be created safely."

# ------------------------------------------------------------
# Verify target counts
# ------------------------------------------------------------

acs_counts = np.bincount(
    acs_target,
    minlength=4
)

print("\nACS CLASS COUNTS")

acs_class_names = [
    "Non-ACS",
    "STEMI",
    "NSTEMI",
    "UA"
]

for i, (name, count) in enumerate(
    zip(acs_class_names, acs_counts)
):
    print(
        f"Class {i} ({name:7s}): "
        f"{count:5d}"
    )


# ============================================================
# 3. STORE 4-CLASS ACS TARGET
# ============================================================

train_dataset.acs_target = acs_target

print("\n✓ 4-class ACS target created")
print("ACS target shape :", train_dataset.acs_target.shape)


# ============================================================
# 2. CALCULATE ACS CLASS WEIGHTS
# ============================================================
#
# ACS is already encoded as ONE 4-class target:
#
# 0 = Non-ACS
# 1 = STEMI
# 2 = NSTEMI
# 3 = UA
#
# train_dataset.acs shape:
# (15273,)
#
# ============================================================

acs_targets = train_dataset.acs

print("\nACS TARGET CHECK")
print("----------------")
print("Shape:", acs_targets.shape)
print("Unique classes:", np.unique(acs_targets))

# ------------------------------------------------------------
# CLASS COUNTS
# ------------------------------------------------------------

acs_counts = np.bincount(
    acs_targets,
    minlength=4
)

# ------------------------------------------------------------
# CLASS WEIGHTS
# ------------------------------------------------------------

acs_weights = (
    len(acs_targets)
    / (4 * acs_counts)
)

acs_weights = torch.tensor(
    acs_weights,
    dtype=torch.float32,
    device=device
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

acs_names = [
    "Non-ACS",
    "STEMI",
    "NSTEMI",
    "UA"
]

print("\nACS CLASS COUNTS")
print("----------------")

for i, (name, count) in enumerate(
    zip(acs_names, acs_counts)
):
    print(
        f"Class {i} ({name:8s}): "
        f"{count:5d} | "
        f"Weight: {acs_weights[i].item():.4f}"
    )

print("\n✓ ACS is correctly configured as a 4-class target")


# ============================================================
# 5. SEVERITY CLASS WEIGHTS
# ============================================================
#
# 0 = None
# 1 = OMI
# 2 = CTO
#
# ============================================================

severity_targets = np.zeros(
    len(train_dataset),
    dtype=np.int64
)

omi = train_dataset.severity[:, 0]
cto = train_dataset.severity[:, 1]

# OMI
severity_targets[omi == 1] = 1

# CTO
severity_targets[cto == 1] = 2

# Check overlap
severity_sum = (
    train_dataset.severity[:, 0]
    + train_dataset.severity[:, 1]
)

multiple_severity = np.sum(
    severity_sum > 1
)

print("\nSEVERITY TARGET VERIFICATION")
print("----------------------------")
print(
    "Samples with both OMI and CTO :",
    multiple_severity
)

assert multiple_severity == 0, \
    "Some samples contain both OMI and CTO."

train_dataset.severity_target = severity_targets

severity_counts = np.bincount(
    severity_targets,
    minlength=3
)

severity_weights = (
    len(train_dataset)
    / (3 * severity_counts)
)

severity_weights = torch.tensor(
    severity_weights,
    dtype=torch.float32,
    device=device
)

print("\nSEVERITY CLASS COUNTS")

severity_names = [
    "None",
    "OMI",
    "CTO"
]

for i, (name, count) in enumerate(
    zip(severity_names, severity_counts)
):
    print(
        f"Class {i} ({name:4s}) | "
        f"Count: {count:5d} | "
        f"Weight: {severity_weights[i].item():.4f}"
    )


# ============================================================
# 6. CORONARY CLASS WEIGHTS
# ============================================================
#
# 10 vessels
#
# 0 = none
# 1 = acute
# 2 = chronic
#
# ============================================================

coronary_weights = []

print("\nCORONARY CLASS WEIGHTS")
print("----------------------")

for vessel_index, vessel_name in enumerate(
    CORONARY_LABELS
):

    counts = np.bincount(
        train_dataset.coronary[:, vessel_index],
        minlength=3
    )

    weights = (
        len(train_dataset)
        / (3 * counts)
    )

    weights = torch.tensor(
        weights,
        dtype=torch.float32,
        device=device
    )

    coronary_weights.append(weights)

    print(
        f"{vessel_name:5s} | "
        f"0: {weights[0].item():.3f} | "
        f"1: {weights[1].item():.3f} | "
        f"2: {weights[2].item():.3f}"
    )


# ============================================================
# 7. CONTEXT CLASS WEIGHTS
# ============================================================
#
# Binary labels:
#
# Paced
# Prior_PCI
# VF_VT
#
# ============================================================

context_counts = []

for i, label in enumerate(CONTEXT_LABELS):

    positive = np.sum(
        train_dataset.context[:, i] == 1
    )

    negative = np.sum(
        train_dataset.context[:, i] == 0
    )

    context_counts.append(
        (negative, positive)
    )

print("\nCONTEXT CLASS COUNTS")
print("--------------------")

for label, (negative, positive) in zip(
    CONTEXT_LABELS,
    context_counts
):

    print(
        f"{label:10s} | "
        f"Negative: {negative:5d} | "
        f"Positive: {positive:5d}"
    )


# ============================================================
# 8. CREATE LOSS FUNCTIONS
# ============================================================

# ACS
acs_criterion = nn.CrossEntropyLoss(
    weight=acs_weights
)

# Severity
severity_criterion = nn.CrossEntropyLoss(
    weight=severity_weights
)

# Coronary
coronary_criteria = [
    nn.CrossEntropyLoss(
        weight=weights
    )
    for weights in coronary_weights
]

# Context
context_pos_weight = []

for negative, positive in context_counts:

    weight = negative / max(positive, 1)

    context_pos_weight.append(weight)

context_pos_weight = torch.tensor(
    context_pos_weight,
    dtype=torch.float32,
    device=device
)

context_criterion = nn.BCEWithLogitsLoss(
    pos_weight=context_pos_weight
)


# ============================================================
# 9. OPTIMIZER
# ============================================================
#
# ONLY the four new heads are optimized.
#
# ECGFounder backbone remains frozen.
#
# ============================================================

head_parameters = (
    list(multi_model.acs_head.parameters())
    + list(multi_model.severity_head.parameters())
    + list(multi_model.coronary_head.parameters())
    + list(multi_model.context_head.parameters())
)

optimizer = torch.optim.AdamW(
    head_parameters,
    lr=1e-3,
    weight_decay=1e-4
)


# ============================================================
# 10. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("STAGE 1 TRAINING SETUP")
print("=" * 60)

print("Optimizer       : AdamW")
print("Learning rate   : 0.001")
print("Weight decay    : 0.0001")

print("\nLoss functions:")
print("ACS             : Weighted CrossEntropy")
print("Severity        : Weighted CrossEntropy")
print("Coronary        : Weighted CrossEntropy × 10")
print("Context         : Weighted BCEWithLogits")

print("\nTrainable heads:")
print("ACS             : 4 classes")
print("Severity        : 3 classes")
print("Coronary        : 10 vessels × 3 states")
print("Context         : 3 binary labels")

print("\n" + "=" * 60)
print("✓ STAGE 1 LOSS + OPTIMIZER READY")
print("=" * 60)

Device: cuda
CHECKING ECGFOUNDER FREEZE STATUS
Backbone trainable parameters : 0
Head trainable parameters     : 41000
✓ ECGFounder backbone is frozen
✓ Classifier heads are trainable


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
# ============================================================
# STAGE 1 — TRAIN CLASSIFIER HEADS
# ECGFounder FROZEN
# ============================================================

import torch
import os

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10

BEST_MODEL_PATH = os.path.join(
    BASE,
    "results",
    "models",
    "stage1_heads_best.pth"
)

os.makedirs(
    os.path.dirname(BEST_MODEL_PATH),
    exist_ok=True
)

# ============================================================
# MOVE MODEL TO DEVICE
# ============================================================

multi_model = multi_model.to(device)

best_val_loss = float("inf")


# ============================================================
# FUNCTION: CALCULATE TOTAL LOSS
# ============================================================

def calculate_loss(
    acs,
    severity,
    coronary,
    context,
    acs_target,
    severity_target,
    coronary_target,
    context_target
):

    # --------------------------------------------------------
    # ACS
    # --------------------------------------------------------

    acs_target = torch.argmax(
        acs_target,
        dim=1
    )

    acs_loss = acs_criterion(
        acs,
        acs_target
    )

    # --------------------------------------------------------
    # SEVERITY
    # --------------------------------------------------------

    severity_target = torch.argmax(
        severity_target,
        dim=1
    )

    severity_loss = severity_criterion(
        severity,
        severity_target
    )

    # --------------------------------------------------------
    # CORONARY
    # --------------------------------------------------------

    coronary_loss = 0.0

    for vessel in range(10):

        vessel_loss = coronary_criteria[vessel](
            coronary[:, vessel, :],
            coronary_target[:, vessel]
        )

        coronary_loss += vessel_loss

    coronary_loss = coronary_loss / 10.0

    # --------------------------------------------------------
    # CONTEXT
    # --------------------------------------------------------

    context_loss = context_criterion(
        context,
        context_target
    )

    # --------------------------------------------------------
    # TOTAL
    # --------------------------------------------------------

    total_loss = (
        acs_loss
        + severity_loss
        + coronary_loss
        + context_loss
    )

    return (
        total_loss,
        acs_loss,
        severity_loss,
        coronary_loss,
        context_loss
    )


# ============================================================
# TRAINING LOOP
# ============================================================

print("=" * 70)
print("STAGE 1 — CLASSIFIER HEAD TRAINING")
print("=" * 70)

for epoch in range(EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    multi_model.train()

    # Keep ECGFounder frozen
    multi_model.backbone.eval()

    train_total = 0.0
    train_acs = 0.0
    train_severity = 0.0
    train_coronary = 0.0
    train_context = 0.0

    train_batches = 0

    for (
        ecg,
        acs_target,
        severity_target,
        coronary_target,
        context_target
    ) in train_loader:

        ecg = ecg.to(device)
        acs_target = acs_target.to(device)
        severity_target = severity_target.to(device)
        coronary_target = coronary_target.to(device)
        context_target = context_target.to(device)

        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad()

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        (
            acs,
            severity,
            coronary,
            context
        ) = multi_model(ecg)

        # ----------------------------------------------------
        # Calculate losses
        # ----------------------------------------------------

        (
            total_loss,
            acs_loss,
            severity_loss,
            coronary_loss,
            context_loss
        ) = calculate_loss(
            acs,
            severity,
            coronary,
            context,
            acs_target,
            severity_target,
            coronary_target,
            context_target
        )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        total_loss.backward()

        # ----------------------------------------------------
        # Update ONLY classifier heads
        # ----------------------------------------------------

        optimizer.step()

        # ----------------------------------------------------
        # Accumulate
        # ----------------------------------------------------

        train_total += total_loss.item()
        train_acs += acs_loss.item()
        train_severity += severity_loss.item()
        train_coronary += coronary_loss.item()
        train_context += context_loss.item()

        train_batches += 1

    # ========================================================
    # VALIDATION
    # ========================================================

    multi_model.eval()

    val_total = 0.0
    val_acs = 0.0
    val_severity = 0.0
    val_coronary = 0.0
    val_context = 0.0

    val_batches = 0

    with torch.no_grad():

        for (
            ecg,
            acs_target,
            severity_target,
            coronary_target,
            context_target
        ) in val_loader:

            ecg = ecg.to(device)
            acs_target = acs_target.to(device)
            severity_target = severity_target.to(device)
            coronary_target = coronary_target.to(device)
            context_target = context_target.to(device)

            (
                acs,
                severity,
                coronary,
                context
            ) = multi_model(ecg)

            (
                total_loss,
                acs_loss,
                severity_loss,
                coronary_loss,
                context_loss
            ) = calculate_loss(
                acs,
                severity,
                coronary,
                context,
                acs_target,
                severity_target,
                coronary_target,
                context_target
            )

            val_total += total_loss.item()
            val_acs += acs_loss.item()
            val_severity += severity_loss.item()
            val_coronary += coronary_loss.item()
            val_context += context_loss.item()

            val_batches += 1

    # ========================================================
    # AVERAGE LOSSES
    # ========================================================

    train_total /= train_batches
    train_acs /= train_batches
    train_severity /= train_batches
    train_coronary /= train_batches
    train_context /= train_batches

    val_total /= val_batches
    val_acs /= val_batches
    val_severity /= val_batches
    val_coronary /= val_batches
    val_context /= val_batches

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_total < best_val_loss:

        best_val_loss = val_total

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": multi_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_total
            },
            BEST_MODEL_PATH
        )

        saved = "✓ BEST SAVED"

    else:

        saved = ""

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"\nEpoch [{epoch + 1:02d}/{EPOCHS}]"
    )

    print(
        f"Train Loss: {train_total:.4f} | "
        f"ACS: {train_acs:.4f} | "
        f"Severity: {train_severity:.4f} | "
        f"Coronary: {train_coronary:.4f} | "
        f"Context: {train_context:.4f}"
    )

    print(
        f"Val   Loss: {val_total:.4f} | "
        f"ACS: {val_acs:.4f} | "
        f"Severity: {val_severity:.4f} | "
        f"Coronary: {val_coronary:.4f} | "
        f"Context: {val_context:.4f}"
    )

    if saved:
        print(saved)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("✓ STAGE 1 TRAINING COMPLETE")
print("=" * 70)

print("Best validation loss :", best_val_loss)
print("Best model saved at  :", BEST_MODEL_PATH)

STAGE 1 — CLASSIFIER HEAD TRAINING


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)